In [2]:
import langchain
import chromadb
import streamlit
import sentence_transformers
import pypdf

print("LangChain:", langchain.__version__)
print("ChromaDB:", chromadb.__version__)
print("✅ Everything is working!")

LangChain: 1.3.14
ChromaDB: 1.5.9
✅ Everything is working!


In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader

documents = []

pdf_folder = "data"

for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        print("Loading:", file)
        loader = PyPDFLoader(os.path.join(pdf_folder, file))
        documents.extend(loader.load())

print("\n✅ Total pages loaded:", len(documents))

C:\Users\Sinchana.S.Acharya\AppData\Local\Temp\ipykernel_12724\1826080088.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading: 10min.pdf
Loading: basics.pdf
Loading: merging.pdf
Loading: missing_data.pdf

✅ Total pages loaded: 195


In [5]:
import langchain
print(langchain.__version__)

1.3.14


In [6]:
import langchain_text_splitters
print("langchain_text_splitters is installed")

langchain_text_splitters is installed


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 432


In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ Embedding model loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4185.61it/s]


✅ Embedding model loaded successfully!


In [9]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

client = chromadb.PersistentClient(path="chroma_db")

embedding_function = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="rag_collection",
    embedding_function=embedding_function
)

print("✅ ChromaDB collection created!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3391.85it/s]


✅ ChromaDB collection created!


In [10]:
for i, chunk in enumerate(chunks):
    collection.add(
        ids=[str(i)],
        documents=[chunk.page_content],
        metadatas=[chunk.metadata]
    )

print(f"✅ Stored {len(chunks)} chunks in ChromaDB.")

✅ Stored 432 chunks in ChromaDB.


In [11]:
query = "What is missing data?"

results = collection.query(
    query_texts=[query],
    n_results=5
)

print(results["documents"][0])

['\ueddb\nThe descriptive statistics and computational methods discussed in the data structure overview\n(and listed here and here) all account for missing data.\nWhen summing data, NA values or empty data will be treated as zero.\nWhen taking the product, NA values or empty data will be treated as 1.\nCumulative methods like cumsum() and cumprod() ignore NA values by default, but preserve\nthem in the resulting array. To override this behaviour and include NA values in the calculation,\nuse skipna=False.', '\ueddb\nWorking with missing data\nValues considered “missing”\npandas uses different sentinel values to represent a missing (also referred to as NA) depending\non the data type.\nnumpy.nan for NumPy data types. The disadvantage of using NumPy data types is that the\noriginal data type will be coerced to np.float64 or object.\nNaT for NumPy np.datetime64, np.timedelta64, and PeriodDtype. For typing applications,\nuse api.typing.NaTType.\nIn [1]: pd.Series([1, 2], dtype=np.int64).re

In [12]:
for i, doc in enumerate(results["documents"][0], start=1):
    print("=" * 80)
    print(f"Result {i}")
    print("=" * 80)
    print(doc)
    print()

Result 1

The descriptive statistics and computational methods discussed in the data structure overview
(and listed here and here) all account for missing data.
When summing data, NA values or empty data will be treated as zero.
When taking the product, NA values or empty data will be treated as 1.
Cumulative methods like cumsum() and cumprod() ignore NA values by default, but preserve
them in the resulting array. To override this behaviour and include NA values in the calculation,
use skipna=False.

Result 2

Working with missing data
Values considered “missing”
pandas uses different sentinel values to represent a missing (also referred to as NA) depending
on the data type.
numpy.nan for NumPy data types. The disadvantage of using NumPy data types is that the
original data type will be coerced to np.float64 or object.
NaT for NumPy np.datetime64, np.timedelta64, and PeriodDtype. For typing applications,
use api.typing.NaTType.
In [1]: pd.Series([1, 2], dtype=np.int64).reindex([0, 1,

In [13]:
query = "What is merge in pandas?"

In [14]:
query = "Explain concat."

In [15]:
query = "What are missing values?"

In [16]:
query = "How do I fill null values?"